# Bronze data testing

Đọc thử dữ liệu bronze (`catalog.bronze.adjust_events`) từ Iceberg/MinIO để kiểm tra data đã sink đúng chưa.

Yêu cầu trước khi chạy:
- `make up` đã chạy, MinIO + bronze job đã ghi ít nhất 1 batch.
- Kernel dùng venv `.venv-notebook` (đã cài `pyspark==4.0.0`, `jupyter`, `pandas`, `python-dotenv`).
- Biến môi trường lấy từ `.env` ở project root (không hard-code credentials trong notebook).

In [41]:
import os
from pathlib import Path

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
load_dotenv(PROJECT_ROOT / ".env")

ENV = os.getenv("ENV", "dev")
ICEBERG_CATALOG = os.getenv("ICEBERG_CATALOG", "catalog")
ICEBERG_WAREHOUSE = os.getenv("ICEBERG_WAREHOUSE", f"s3a://lakehouse/{ENV}")
S3_ENDPOINT = os.getenv("S3_ENDPOINT_LOCAL", "http://localhost:9000")
AWS_ACCESS_KEY_ID = os.environ["AWS_ACCESS_KEY_ID"]
AWS_SECRET_ACCESS_KEY = os.environ["AWS_SECRET_ACCESS_KEY"]

JARS_DIR = PROJECT_ROOT / ".spark-jars"
JARS = ",".join(str(p) for p in JARS_DIR.glob("*.jar"))

In [42]:
from pyspark.sql import SparkSession
spark.stop()

spark = SparkSession.builder \
    .appName("BronzeDataTesting") \
    .master("local[*]") \
    .config("spark.jars", JARS) \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config(f"spark.sql.catalog.{ICEBERG_CATALOG}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{ICEBERG_CATALOG}.type", "hadoop") \
    .config(f"spark.sql.catalog.{ICEBERG_CATALOG}.warehouse", ICEBERG_WAREHOUSE) \
    .config("spark.hadoop.fs.s3a.endpoint", S3_ENDPOINT) \
    .config("spark.hadoop.fs.s3a.access.key", AWS_ACCESS_KEY_ID) \
    .config("spark.hadoop.fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

spark

26/09/22 08:12:10 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## Liệt kê bảng trong bronze schema

In [43]:
spark.sql(f"SHOW TABLES IN {ICEBERG_CATALOG}.bronze").show(truncate=False)

+---------+-------------+-----------+
|namespace|tableName    |isTemporary|
+---------+-------------+-----------+
|bronze   |adjust_events|false      |
+---------+-------------+-----------+



## Đọc bảng `adjust_events` (batch, không phải streaming)

In [44]:
bronze_table = f"{ICEBERG_CATALOG}.bronze.adjust_events"
print(bronze_table)
df = spark.table(bronze_table)
df.printSchema()

catalog.bronze.adjust_events
root
 |-- id: integer (nullable = true)
 |-- activity_kind: string (nullable = true)
 |-- created_at: long (nullable = true)
 |-- app_token: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- app_name: string (nullable = true)
 |-- app_version: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- environment: string (nullable = true)
 |-- sdk_version: string (nullable = true)
 |-- os_name: string (nullable = true)
 |-- os_version: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- device_model: string (nullable = true)
 |-- language: string (nullable = true)
 |-- country: string (nullable = true)
 |-- country_subdivision: string (nullable = true)
 |-- city: string (nullable = true)
 |-- timezone: string (nullable = true)
 |-- adid: string (nullable = true)
 |-- gps_adid: string (nullable = true)
 |-- idfa: string (nullable = true)
 |-- idfv: string (nullable = true)
 |-- tracker: string (nullable = t

In [45]:
print("Row count:", df.count())
df.orderBy(df.created_at.desc()).limit(20).toPandas()

Row count: 10427


,id,activity_kind,created_at,app_token,store_id,app_name,app_version,platform,environment,sdk_version,...,subscription_product_id,subscription_sales_region,reporting_cost,installed_at,click_time,impression_time,engagement_time,impression_based,is_organic,event_date
0,10385,ad_revenue,1790089918348907,app_d1clwce6w415,com.example.example3,Example Three,1.1.0,ios,production,4.38.0,...,NaN,NaN,None,NaN,NaN,NaN,NaN,None,None,2026-09-22
1,10377,ad_revenue,1790089918348907,app_d1clwce6w415,com.example.example3,Example Three,1.1.0,ios,production,4.38.0,...,NaN,NaN,None,NaN,NaN,NaN,NaN,None,None,2026-09-22
2,10352,ad_revenue,1790089918348907,app_d1clwce6w415,com.example.example3,Example Three,1.1.0,ios,production,4.38.0,...,NaN,NaN,None,NaN,NaN,NaN,NaN,None,None,2026-09-22
3,10353,ad_revenue,1790089918348907,app_d1clwce6w415,com.example.example3,Example Three,1.1.0,ios,production,4.38.0,...,NaN,NaN,None,NaN,NaN,NaN,NaN,None,None,2026-09-22
4,10355,ad_revenue,1790089918348907,app_d1clwce6w415,com.example.example3,Example Three,1.1.0,ios,production,4.38.0,...,NaN,NaN,None,NaN,NaN,NaN,NaN,None,None,2026-09-22
5,10362,ad_revenue,1790089918348907,app_d1clwce6w415,com.example.example3,Example Three,1.1.0,ios,production,4.38.0,...,NaN,NaN,None,NaN,NaN,NaN,NaN,None,None,2026-09-22
6,10358,ad_revenue,1790089918348907,app_d1clwce6w415,com.example.example3,Example Three,1.1.0,ios,production,4.38.0,...,NaN,NaN,None,NaN,NaN,NaN,NaN,None,None,2026-09-22
7,10369,ad_revenue,1790089918348907,app_d1clwce6w415,com.example.example3,Example Three,1.1.0,ios,production,4.38.0,...,NaN,NaN,None,NaN,NaN,NaN,NaN,None,None,2026-09-22
8,10373,ad_revenue,1790089918348907,app_d1clwce6w415,com.example.example3,Example Three,1.1.0,ios,production,4.38.0,...,NaN,NaN,None,NaN,NaN,NaN,NaN,None,None,2026-09-22
9,10378,ad_revenue,1790089918348907,app_d1clwce6w415,com.example.example3,Example Three,1.1.0,ios,production,4.38.0,...,NaN,NaN,None,NaN,NaN,NaN,NaN,None,None,2026-09-22


In [ ]:
# spark.stop()